# Session 1: OpenAI API Engineering with Structured Outputs (Student Notebook)

Welcome to Session 1 of Day 2! In this notebook, you will move beyond basic chat completions into production-grade API engineering. You will master structured output generation using JSON mode, function calling (tool use), and Pydantic validation -- skills essential for building reliable agentic systems that produce machine-readable outputs.

**McKinsey Context:** All demos and tasks use real-world consulting scenarios -- strategy engagements, M&A due diligence, digital transformation assessments, market sizing exercises, and CEO briefings -- to demonstrate how structured outputs power consulting workflows at scale.

## Learning Objectives

By the end of this session, you will be able to:

1. **Use streaming** and track token usage for cost management
2. **Generate structured JSON** outputs using `response_format` parameter
3. **Define and use function calling** (tool use) with the OpenAI API
4. **Validate LLM outputs** with Pydantic models for type safety
5. **Build extraction pipelines** that turn unstructured text into structured data
6. **Implement robust API wrappers** with retries and error handling

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# ============================================================
# Imports and Setup
# ============================================================

import openai
import json
import os
from pydantic import BaseModel, Field
from typing import Optional, List

# Ensure your OpenAI API key is set as an environment variable:
#   export OPENAI_API_KEY="sk-..."
# Or uncomment the line below and paste your key (not recommended for production):
# os.environ["OPENAI_API_KEY"] = "sk-..."

print("All imports successful!")
print(f"OpenAI SDK version: {openai.__version__}")

---
## Demos (Follow Along)

The following five demos are fully coded. Run each cell, observe the output, and make sure you understand what is happening before moving on.

### Demo 1: OpenAI API Deep Dive — Streaming, Token Usage, and Finish Reasons

In production consulting systems you need to: (a) track token usage for engagement cost management, (b) stream responses for real-time partner briefings, and (c) inspect finish reasons to know if the model completed its analysis or was cut off.

**McKinsey Scenario:** A Partner asks the AI assistant to generate a concise market overview for a CEO briefing. We track tokens to manage costs across multiple client engagements and stream the response so the delivery team can review output in real time.

In [ ]:
# Demo 1 - OpenAI API Deep Dive (McKinsey Consulting Context)

client = openai.OpenAI()

# Part A: Token usage tracking — generating a market overview for a CEO briefing
response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "Provide a 3-sentence executive summary of the global healthcare market outlook for a McKinsey CEO briefing."}],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "150"))
)

print("=== Token Usage (Engagement Cost Tracking) ===")
print(f"Prompt tokens    : {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens     : {response.usage.total_tokens}")
print(f"Finish reason    : {response.choices[0].finish_reason}")
print(f"\nCEO Briefing Response:\n{response.choices[0].message.content}")

# Part B: Streaming responses — real-time delivery for partner review
print("\n=== Streaming Response (Partner Review) ===")
stream = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "List the top 3 priorities for a digital transformation engagement at a Fortune 500 retailer, one per line."}],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "100")),
    stream=True
)

collected_text = ""
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        token = chunk.choices[0].delta.content
        collected_text += token
        print(token, end="", flush=True)

print(f"\n\nFull collected text: {collected_text}")

### Demo 2: Structured Output with JSON Mode — Client Profile Extraction

By setting `response_format={"type": "json_object"}`, the model is guaranteed to return valid JSON. This is critical for consulting workflows where downstream systems must parse client data reliably. You **must** include the word "JSON" in your prompt when using this mode.

**McKinsey Scenario:** An Engagement Manager needs a structured client profile for a new healthcare engagement. JSON mode ensures the profile data is machine-readable for the firm's CRM and engagement tracking systems.

In [ ]:
# Demo 2 - Structured Output with JSON Mode (McKinsey Client Profile)

client = openai.OpenAI()

response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": "You are a McKinsey engagement assistant that outputs JSON."},
        {"role": "user", "content": "Create a client profile for a new engagement with Acme Healthcare Corp. Include: client_name, industry, annual_revenue_usd, number_of_employees, headquarters, key_challenges (list), and engagement_type."}
    ],
    response_format={"type": "json_object"},
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
)

raw_json = response.choices[0].message.content
print("Raw JSON response:")
print(raw_json)

# Parse and pretty-print
parsed = json.loads(raw_json)
print("\nParsed and formatted:")
print(json.dumps(parsed, indent=2))

# Access individual fields
print(f"\nClient      : {parsed.get('client_name', 'N/A')}")
print(f"Industry    : {parsed.get('industry', 'N/A')}")
print(f"Engagement  : {parsed.get('engagement_type', 'N/A')}")

### Demo 3: Function Calling — Consulting Research Tools

Function calling lets the model decide **when** and **how** to call external tools. In a consulting context, this enables an AI assistant to autonomously pull market data, run benchmarks, or look up competitor intelligence. The flow is:
1. You define tool schemas (e.g., `market_research`, `competitor_lookup`) and send them with the request
2. The model returns a `tool_calls` response (instead of regular content)
3. You execute the function locally (simulated data)
4. You send the result back to the model for a final answer

**McKinsey Scenario:** An Associate asks the AI for market research on a specific industry. The model decides to call the `market_research` tool to retrieve structured market data.

In [ ]:
# Demo 3 - Function Calling / Tool Use (McKinsey Consulting Research)

client = openai.OpenAI()

# Step 1: Define the tools (functions) the model can call
tools = [
    {
        "type": "function",
        "function": {
            "name": "market_research",
            "description": "Get market research data for a specific industry sector",
            "parameters": {
                "type": "object",
                "properties": {
                    "industry": {
                        "type": "string",
                        "description": "The industry sector, e.g., 'healthcare', 'financial_services', 'energy'"
                    },
                    "region": {
                        "type": "string",
                        "enum": ["north_america", "europe", "asia_pacific", "global"],
                        "description": "Geographic region for the analysis"
                    }
                },
                "required": ["industry"]
            }
        }
    }
]

# Step 2: Simulated market research function
def market_research(industry, region="global"):
    """Simulated market research function returning industry data."""
    market_data = {
        "healthcare": {"market_size_usd_bn": 8450, "cagr_pct": 7.9, "key_trend": "AI-driven diagnostics and personalized medicine"},
        "financial_services": {"market_size_usd_bn": 28500, "cagr_pct": 6.2, "key_trend": "Embedded finance and real-time payments"},
        "energy": {"market_size_usd_bn": 6700, "cagr_pct": 8.5, "key_trend": "Energy transition and grid modernization"},
    }
    data = market_data.get(industry, {"market_size_usd_bn": 5000, "cagr_pct": 5.0, "key_trend": "Digital transformation"})
    return json.dumps({"industry": industry, "region": region, "market_size_usd_bn": data["market_size_usd_bn"], "cagr_pct": data["cagr_pct"], "key_trend": data["key_trend"]})

# Step 3: Make the initial API call
response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "What does the healthcare market look like globally? I need data for a client engagement."}],
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message
print(f"Finish reason: {response.choices[0].finish_reason}")

# Step 4: Check if the model wants to call a function
if message.tool_calls:
    tool_call = message.tool_calls[0]
    print(f"Function called: {tool_call.function.name}")
    print(f"Arguments: {tool_call.function.arguments}")

    # Step 5: Execute the function
    args = json.loads(tool_call.function.arguments)
    result = market_research(**args)
    print(f"Function result: {result}")

    # Step 6: Send the result back to the model
    follow_up = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "user", "content": "What does the healthcare market look like globally? I need data for a client engagement."},
            message,
            {"role": "tool", "tool_call_id": tool_call.id, "content": result}
        ],
        tools=tools
    )
    print(f"\nFinal response:\n{follow_up.choices[0].message.content}")
else:
    print(f"Direct response: {message.content}")

### Demo 4: Pydantic-Based Response Validation — Engagement Summary

Pydantic models let you define the **exact shape** of the data you expect from the LLM, with type checking and constraints. If the LLM returns data that does not match, Pydantic will raise an error -- catching problems before they propagate through your consulting workflow.

**McKinsey Scenario:** Validate a structured engagement summary for a recently completed strategy engagement. The Pydantic model enforces that all required fields (client name, industry, engagement type, satisfaction score, etc.) are present and correctly typed.

In [ ]:
# Demo 4 - Pydantic-Based Response Validation (McKinsey Engagement Summary)

client = openai.OpenAI()

# Step 1: Define Pydantic models for structured consulting output
class EngagementSummary(BaseModel):
    client_name: str = Field(description="Name of the client organization")
    industry: str = Field(description="Client industry sector")
    engagement_type: str = Field(description="Type of engagement, e.g., Strategy, M&A Due Diligence, Digital Transformation")
    duration_weeks: int = Field(description="Duration of the engagement in weeks")
    satisfaction_score: float = Field(ge=1.0, le=5.0, description="Client satisfaction score (1-5)")
    key_recommendation: str = Field(description="One-sentence primary recommendation")
    follow_on_opportunity: bool = Field(description="Whether there is a follow-on engagement opportunity")

# Step 2: Request structured output using JSON mode
response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": (
            "You are a McKinsey engagement tracking system. Output your engagement summary as JSON with these exact fields: "
            "client_name (string), industry (string), engagement_type (string), duration_weeks (integer), "
            "satisfaction_score (float 1-5), key_recommendation (string), follow_on_opportunity (boolean)."
        )},
        {"role": "user", "content": "Summarize the recently completed digital transformation engagement with Meridian Financial Group."}
    ],
    response_format={"type": "json_object"},
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
)

raw = response.choices[0].message.content
print("Raw JSON:")
print(raw)

# Step 3: Validate with Pydantic
try:
    summary = EngagementSummary.model_validate_json(raw)
    print("\nValidated EngagementSummary:")
    print(f"  Client          : {summary.client_name}")
    print(f"  Industry        : {summary.industry}")
    print(f"  Engagement Type : {summary.engagement_type}")
    print(f"  Duration        : {summary.duration_weeks} weeks")
    print(f"  Satisfaction    : {summary.satisfaction_score}/5.0")
    print(f"  Recommendation  : {summary.key_recommendation}")
    print(f"  Follow-on       : {'Yes' if summary.follow_on_opportunity else 'No'}")
except Exception as e:
    print(f"\nValidation error: {e}")

### Demo 5: Building a Structured Data Extraction Pipeline — Engagement Team Extraction

This demo combines JSON mode and Pydantic into a reusable pipeline that extracts structured consulting team information from unstructured engagement notes. This pattern is the foundation for many agentic data-processing workflows at McKinsey -- from staffing databases to knowledge management systems.

**McKinsey Scenario:** An engagement kickoff email contains team member details in unstructured text. We build a pipeline to extract each team member's name, role, email, practice area, and office location into structured data.

In [ ]:
# Demo 5 - Building a Structured Data Extraction Pipeline (McKinsey Engagement Team)

client = openai.OpenAI()

class TeamMember(BaseModel):
    name: str = Field(description="Full name of the team member")
    email: Optional[str] = Field(default=None, description="Email address if found")
    phone: Optional[str] = Field(default=None, description="Phone number if found")
    role: Optional[str] = Field(default=None, description="Consulting role: Partner, Associate Partner, Engagement Manager, Associate, Business Analyst")
    practice: Optional[str] = Field(default=None, description="McKinsey practice area if found")

def extract_team_members(text: str) -> List[TeamMember]:
    """Extract structured team member information from unstructured engagement notes."""
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": (
                "Extract all consulting team member information from the text. "
                "Return a JSON object with a 'team_members' key containing a list. "
                "Each member should have: name, email (null if not found), "
                "phone (null if not found), role (null if not found), "
                "practice (null if not found)."
            )},
            {"role": "user", "content": text}
        ],
        response_format={"type": "json_object"},
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "500"))
    )

    data = json.loads(response.choices[0].message.content)
    members = []
    for item in data.get("team_members", []):
        try:
            members.append(TeamMember(**item))
        except Exception as e:
            print(f"Skipping invalid team member: {e}")
    return members

# Test the extraction pipeline with an engagement staffing email
sample_text = """
Team, here is the staffing for the Apex Energy transformation engagement:

The engagement will be led by Partner Rajesh Gupta from our Houston office (Energy Practice).
Reach him at rajesh.gupta@mckinsey.com or (713) 555-0142.

Engagement Manager Sarah Okonkwo from the Chicago office will run day-to-day operations.
She is part of our Operations Practice. Her email is sarah.okonkwo@mckinsey.com.

We also have two Associates staffed: David Chen (Digital/McKinsey Digital, New York office,
david.chen@mckinsey.com) and Priya Sharma (Strategy & Corporate Finance, London office).

For analytics support, reach out to Business Analyst Tom Williams at tom.williams@mckinsey.com.
"""

print("Input text:")
print(sample_text)
print("=" * 60)
print("Extracted engagement team:")
print("=" * 60)

members = extract_team_members(sample_text)
for i, member in enumerate(members, 1):
    print(f"\nTeam Member {i}:")
    print(f"  Name     : {member.name}")
    print(f"  Email    : {member.email or 'N/A'}")
    print(f"  Phone    : {member.phone or 'N/A'}")
    print(f"  Role     : {member.role or 'N/A'}")
    print(f"  Practice : {member.practice or 'N/A'}")

---
## Tasks (Your Turn!)

Now it is your turn to practice. Each task below has a description, hints, and a code skeleton with `TODO` placeholders. Fill in the code to complete each task. All tasks use McKinsey consulting scenarios.

### Task 1: Build a Structured Competitive Assessment Extractor Using JSON Mode

Create a function that takes an M&A due diligence briefing (string) and extracts structured competitive intelligence -- companies, executives, financial metrics, strategic actions, and industries -- returning the result as a parsed dictionary. This mirrors how a McKinsey Associate would structure findings from unstructured deal memos in a MECE (Mutually Exclusive, Collectively Exhaustive) format.

In [ ]:
# Task 1 - Build a Structured Competitive Assessment Extractor Using JSON Mode

# TODO: Create a function that takes an M&A due diligence briefing (string) and extracts
# structured competitive intelligence: companies, executives, financial_metrics,
# strategic_actions, and industries.
# Return the result as a parsed dictionary.
#
# Hint: Use response_format={"type": "json_object"}
# Hint: In the system message, act as a "McKinsey M&A due diligence analyst" and define
#       the expected JSON schema with keys: companies, executives, financial_metrics,
#       strategic_actions, industries (all lists of strings)
# Hint: Use json.loads() to parse the response

def extract_competitive_intelligence(briefing_text):
    """Extract competitive intelligence entities from an M&A due diligence briefing using JSON mode."""
    # YOUR CODE HERE
    pass


# Test your function with this M&A briefing
# sample_briefing = (
#     "In Q3 2024, Meridian Health Systems (CEO: Dr. Amanda Foster) announced a $2.3B acquisition "
#     "of Pacific Diagnostics, a move that positions them as the third-largest diagnostic network in "
#     "North America. Goldman Sachs advised on the deal. Meanwhile, competitor UnitedHealth Group "
#     "(CEO: Andrew Witty) reported $92.9B in quarterly revenue and signaled interest in expanding "
#     "its Optum Health division through further acquisitions in the telehealth space. "
#     "McKinsey Partner Lisa Chen is leading the due diligence workstream for a private equity client "
#     "evaluating a potential counter-bid. The target company operates across healthcare and life sciences."
# )
# entities = extract_competitive_intelligence(sample_briefing)
# print(json.dumps(entities, indent=2))

### Task 2: Implement a Financial Analysis Tool with Function Calling

Create a financial analysis tool that uses OpenAI function calling. Define a `financial_analysis` tool with parameters for analysis type (valuation, profitability, leverage, liquidity) and a company name. Implement the actual function with simulated data, and handle the full function-calling loop. This mirrors how a McKinsey Associate would query financial databases during an engagement.

In [ ]:
# Task 2 - Implement a Financial Analysis Tool with Function Calling

# TODO: Create a financial analysis tool that uses OpenAI function calling.
# 1. Define a "financial_analysis" tool with parameters: analysis_type
#    (enum: valuation/profitability/leverage/liquidity) and company (string)
# 2. Implement the actual financial_analysis function with simulated data for
#    companies like "Meridian Health" and "Apex Energy"
# 3. Handle the full function calling loop (request -> tool call -> execute -> respond)
#
# Hint: Define the tool schema with analysis_type as an enum
# Hint: Store simulated financial data in a nested dictionary
#       e.g., {"Meridian Health": {"valuation": {"ev_ebitda": 14.2, ...}, ...}}
# Hint: Parse the function arguments with json.loads()
# Hint: Send the tool result back as a follow-up message with role="tool"

def financial_analysis(analysis_type, company):
    """Perform a financial analysis on a company (simulated data)."""
    # YOUR CODE HERE
    pass

def ask_financial_question(question):
    """Send a financial analysis question and handle function calling."""
    # YOUR CODE HERE
    pass


# Test your function
# print(ask_financial_question("What is the valuation profile of Meridian Health?"))
# print()
# print(ask_financial_question("How profitable is Apex Energy? Give me the margin analysis."))

### Task 3: Create a Multi-Tool Consulting Agent with Automatic Tool Dispatch

Build a consulting research agent that has access to multiple tools (market_research, financial_analysis, benchmarking) and automatically dispatches to the right one based on the user's question. This mirrors how a McKinsey engagement team would leverage multiple knowledge sources during a strategy engagement.

In [ ]:
# Task 3 - Create a Multi-Tool Consulting Agent with Automatic Tool Dispatch

# TODO: Build a consulting research agent that has access to multiple tools and
# automatically dispatches to the right one based on the user's question.
# Tools to implement:
#   1. market_research(industry, region) - returns market size, CAGR, key trends
#   2. financial_analysis(company, metric) - returns specific financial metrics
#   3. benchmarking(company, peer_group) - compares company vs. peer group averages
#
# Hint: Define all three tools in the tools list with appropriate parameter schemas
# Hint: Create a dispatch dictionary mapping function names to implementations
# Hint: Use a system message like "You are a McKinsey consulting research assistant"
# Hint: Handle the case where the model calls a tool AND the case where it responds directly

tools = [
    # YOUR TOOL DEFINITIONS HERE
]

def dispatch_tool_call(tool_name, arguments):
    """Route a tool call to the correct consulting function."""
    # YOUR CODE HERE
    pass

def multi_tool_consulting_agent(user_message):
    """Process a user message with multi-tool consulting support."""
    # YOUR CODE HERE
    pass


# Test your agent
# print("--- Market Research Query ---")
# print(multi_tool_consulting_agent("What is the market size and growth rate for the healthcare industry?"))
# print("\n--- Financial Analysis Query ---")
# print(multi_tool_consulting_agent("What is the EBITDA margin for Apex Energy?"))
# print("\n--- Benchmarking Query ---")
# print(multi_tool_consulting_agent("How does Meridian Health compare to its healthcare peer group?"))

### Task 4: Build a Robust Consulting API Client with Retries, Validation, and Streaming

Build a production-grade API client class that wraps OpenAI chat completions with automatic retry logic, Pydantic validation for structured consulting outputs (e.g., ClientProfile), streaming support for real-time CEO briefings, and token usage tracking for engagement cost management.

In [ ]:
# Task 4 - Build a Robust Consulting API Client with Retries, Validation, and Streaming

# TODO: Build a production-grade consulting API client class that:
# 1. Wraps OpenAI chat completions with automatic retry logic (max 3 retries)
# 2. Validates responses against a Pydantic model (e.g., ClientProfile) when provided
# 3. Supports both streaming (for CEO briefings) and non-streaming modes
# 4. Logs token usage for engagement cost tracking
#
# Hint: Use a try/except loop with exponential backoff (time.sleep(2 ** attempt))
# Hint: Use Pydantic's model_validate_json() for validation
# Hint: Track cumulative token usage in instance variables
# Hint: When response_model is provided, set response_format={"type": "json_object"}

import time

class RobustConsultingClient:
    def __init__(self, model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), max_retries=3):
        """Initialize the consulting client with retry settings."""
        # YOUR CODE HERE
        pass

    def call(self, messages, response_model=None, stream=False, **kwargs):
        """Make a robust API call with retries and optional Pydantic validation."""
        # YOUR CODE HERE
        pass

    def get_usage_stats(self):
        """Return cumulative token usage statistics for engagement cost tracking."""
        # YOUR CODE HERE
        pass


# Test your client with a McKinsey scenario
# consulting_client = RobustConsultingClient()
#
# # Define a ClientProfile Pydantic model
# class ClientProfile(BaseModel):
#     client_name: str
#     industry: str
#     engagement_type: str
#     annual_revenue_bn: float
#
# result = consulting_client.call(
#     messages=[
#         {"role": "system", "content": "Output JSON with fields: client_name, industry, engagement_type, annual_revenue_bn"},
#         {"role": "user", "content": "Create a client profile for Apex Energy, an oil and gas company with $38.7B revenue. We are doing an energy transition strategy engagement."}
#     ],
#     response_model=ClientProfile
# )
# print(f"Validated: {result.client_name} ({result.industry}) - {result.engagement_type}, ${result.annual_revenue_bn}B revenue")
# print(f"Usage: {consulting_client.get_usage_stats()}")
#
# # Test streaming — CEO Briefing
# print("\nStreaming CEO Briefing:")
# consulting_client.call(
#     messages=[{"role": "user", "content": "Provide a 2-sentence executive summary of the key recommendation for a digital transformation at a mid-size retailer."}],
#     stream=True,
#     max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "100"))
# )
# print(f"Usage after streaming: {consulting_client.get_usage_stats()}")

---
## Summary

In this session you learned production-grade OpenAI API engineering skills:

1. **API Deep Dive** -- Streaming responses for real-time output, tracking token usage for cost management, and understanding finish reasons.
2. **JSON Mode** -- Using `response_format={"type": "json_object"}` to get reliable, parseable structured data from the model.
3. **Function Calling** -- Defining tools that the model can invoke, handling the request-execute-respond loop that powers agentic tool use.
4. **Pydantic Validation** -- Using Pydantic models to validate and parse LLM outputs with type safety and automatic error detection.
5. **Extraction Pipelines** -- Combining JSON mode and Pydantic into reusable pipelines that extract structured data from unstructured text.

**Up Next:** In Session 2, we will move from raw API calls to LangChain, learning how to compose modular chains, integrate tools, and build retrieval-augmented generation (RAG) systems.